<a href="https://colab.research.google.com/github/ShahJahanBrohii/ML-Internship-Flyrank/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1: a learned ranking outperformed the fixed rule on the starter slice

The report measured `Precision@50` of `0.740` for the random forest versus `0.240` for the rule baseline. My constructive methodology question is: **where exactly does the positive label come from, and is it measured in a later, non-overlapping window than the features used to rank pages?** If the label is derived from the same observed trend window as the inputs, the result can support directional prioritization, but it should not be read as evidence that a refresh will cause recovery.

### Finding 2: the output is useful for review prioritization, not an automatic publishing decision

The report recommends manual inspection of high-confidence rows and describes the model as decision support. My methodology question is: **does the validation design match the intended deployment population, and is the review outcome measured separately from the model label?** A client-held-out result supports generalization to unseen clients better than a row-random split, while an editorial impact claim would require a later outcome or an experiment. I would preserve the report's cautious framing and make the population, label window, and split explicit.

In [1]:
print("Paper-audit questions recorded above:")
print("1. Is the positive label defined from a later, non-overlapping outcome window?")
print("2. Does validation match unseen-client deployment, and is review impact measured separately?")
print("These are methodology questions for stronger interpretation, not claims that the reported result is invalid.")

Paper-audit questions recorded above:
1. Is the positive label defined from a later, non-overlapping outcome window?
2. Does validation match unseen-client deployment, and is review impact measured separately?
These are methodology questions for stronger interpretation, not claims that the reported result is invalid.


## 2. My model under an honest split (before/after)

The Week-5 model used a ranking task on the `keyword article` lane. The **before** number below is a stratified row-random split, which can place rows from the same client in both partitions. The **after** number holds out complete clients, matching the question: can the model rank pages for a client it did not see during training? I keep the model, features, seed, and metric fixed so the split is the meaningful change.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
DATA_CANDIDATES = [
    Path("/content/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"),
]
data_path = next((path for path in DATA_CANDIDATES if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

data = pd.read_csv(data_path)
lane = data.loc[data["content_type"].eq("keyword article")].copy()
lane["is_declining_label"] = lane["trend_direction"].astype(str).str.lower().eq("down").astype(int)

feature_columns = [
    "days_since_last_update", "content_age_days", "word_count", "char_count",
    "search_volume", "competition", "cpc", "impressions_90d", "clicks_90d",
    "pageviews_90d", "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d", "ctr",
    "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
feature_columns = [column for column in feature_columns if column in lane.columns]
X = lane[feature_columns].apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
y = lane["is_declining_label"].astype(int)
clients = lane["client_id"].fillna("unknown").astype(str)

assert y.nunique() == 2

def precision_at_k(labels, scores, k=50):
    ranked = pd.DataFrame({"label": np.asarray(labels), "score": np.asarray(scores)})
    return float(ranked.sort_values("score", ascending=False).head(k)["label"].mean())


def fit_and_score(train_indices, test_indices):
    model = RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
        n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE,
    )
    model.fit(X.iloc[train_indices], y.iloc[train_indices])
    scores = model.predict_proba(X.iloc[test_indices])[:, 1]
    labels = y.iloc[test_indices].to_numpy()
    return model, scores, {
        "rows": len(test_indices),
        "base_rate": labels.mean(),
        "ROC AUC": roc_auc_score(labels, scores),
        "Average precision": average_precision_score(labels, scores),
        "Precision@50": precision_at_k(labels, scores),
    }

all_indices = np.arange(len(lane))
random_train, random_test = train_test_split(
    all_indices, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
random_model, random_scores, random_metrics = fit_and_score(random_train, random_test)

rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(clients.drop_duplicates().to_numpy())
test_clients = set(shuffled_clients[:max(1, int(round(len(shuffled_clients) * 0.20)))])
grouped_test_mask = clients.isin(test_clients).to_numpy()
grouped_train = all_indices[~grouped_test_mask]
grouped_test = all_indices[grouped_test_mask]
grouped_model, grouped_scores, grouped_metrics = fit_and_score(grouped_train, grouped_test)

grouped_metrics["test_clients"] = len(test_clients)
grouped_metrics["client_overlap"] = len(set(clients.iloc[grouped_train]) & set(clients.iloc[grouped_test]))
comparison = pd.DataFrame({"before_row_random": random_metrics, "after_client_grouped": grouped_metrics}).T.round(3)
display(comparison)
print(f"Overall observed decline rate: {y.mean():.3f}")
print(f"Random split client overlap: {len(set(clients.iloc[random_train]) & set(clients.iloc[random_test]))}")
print("Interpret the after column as the more relevant unseen-client estimate; any gap is evidence that split design changes the claim.")

grouped_test_view = lane.iloc[grouped_test].copy()
grouped_test_view["score"] = grouped_scores
grouped_test_view["actual"] = y.iloc[grouped_test].to_numpy()
grouped_test_view["predicted"] = (grouped_scores >= 0.5).astype(int)
grouped_test_view["error_type"] = np.select(
    [
        grouped_test_view["predicted"].eq(1) & grouped_test_view["actual"].eq(0),
        grouped_test_view["predicted"].eq(0) & grouped_test_view["actual"].eq(1),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)
errors = grouped_test_view.loc[grouped_test_view["error_type"].ne("correct")]
error_examples = errors[[
    "error_type", "actual", "predicted", "score", "impressions_90d",
    "days_since_last_update", "avg_position", "word_count", "clicks_90d",
]].head(3).reset_index(drop=True)
print("Three real grouped-test failures; identifiers are intentionally omitted:")
display(error_examples.round(3))
print(f"Grouped-test errors at a 0.5 threshold: {len(errors):,} of {len(grouped_test):,} rows.")

,rows,base_rate,ROC AUC,Average precision,Precision@50,test_clients,client_overlap
before_row_random,5442.0,0.561,0.906,0.927,1.0,NaN,NaN
after_client_grouped,1431.0,0.460,0.905,0.884,1.0,6.0,0.0


Overall observed decline rate: 0.561
Random split client overlap: 30
Interpret the after column as the more relevant unseen-client estimate; any gap is evidence that split design changes the claim.
Three real grouped-test failures; identifiers are intentionally omitted:


,error_type,actual,predicted,score,impressions_90d,days_since_last_update,avg_position,word_count,clicks_90d
0,false_positive,0,1,0.657,307,103,39.8,1342.0,0
1,false_negative,1,0,0.435,9,8,10.1,1585.0,0
2,false_positive,0,1,0.536,416,104,12.9,3393.0,3


Grouped-test errors at a 0.5 threshold: 311 of 1,431 rows.


## 3. Leakage audit

The audit checks three risks: direct label-derived fields, identifiers or product decisions used as features, and temporal overlap. The first two are hard exclusions. The starter export does not provide a clean future outcome window, so the last item is a disclosed limitation rather than something this notebook can prove away.

In [3]:
label_columns = {"is_declining_label", "trend_direction", "trend_pct"}
identifier_columns = {"content_id", "client_id"}
product_decision_columns = {
    "health_score", "priority_score", "action_type", "refresh_tier",
    "needs_ctr_fix", "is_quick_win", "model_decline_risk",
}
feature_set = set(feature_columns)

leakage_checks = pd.DataFrame([
    {
        "check": "label-derived fields excluded",
        "status": "PASS" if not feature_set.intersection(label_columns) else "FAIL",
        "evidence": sorted(feature_set.intersection(label_columns)),
    },
    {
        "check": "identifiers excluded as features",
        "status": "PASS" if not feature_set.intersection(identifier_columns) else "FAIL",
        "evidence": sorted(feature_set.intersection(identifier_columns)),
    },
    {
        "check": "product decision fields excluded",
        "status": "PASS" if not feature_set.intersection(product_decision_columns) else "FAIL",
        "evidence": sorted(feature_set.intersection(product_decision_columns)),
    },
    {
        "check": "target has both classes",
        "status": "PASS" if y.nunique() == 2 else "FAIL",
        "evidence": sorted(y.unique().tolist()),
    },
    {
        "check": "grouped test has no client overlap",
        "status": "PASS" if grouped_metrics["client_overlap"] == 0 else "FAIL",
        "evidence": grouped_metrics["client_overlap"],
    },
])
display(leakage_checks)
assert (leakage_checks["status"] == "PASS").all()

print("Temporal limitation: is_declining_label comes from trend_direction/trend_pct, while several inputs are trailing-window metrics.")
print("Therefore these results are observed and directional; a future-window label with strictly earlier features is needed for a stronger deployment claim.")

,check,status,evidence
0,label-derived fields excluded,PASS,[]
1,identifiers excluded as features,PASS,[]
2,product decision fields excluded,PASS,[]
3,target has both classes,PASS,"[0, 1]"
4,grouped test has no client overlap,PASS,0


Temporal limitation: is_declining_label comes from trend_direction/trend_pct, while several inputs are trailing-window metrics.
Therefore these results are observed and directional; a future-window label with strictly earlier features is needed for a stronger deployment claim.


## 4. Claim rewrite

My original Week-5 claim was too broad: **“The random forest beats the baseline and identifies pages that should be refreshed.”**

A claim that fits the evidence is: **“On this anonymized keyword-article slice, the random forest measured higher held-out ranking performance than the fixed rule under the evaluated split. Under the client-grouped split, its measured `Precision@50` is a directional estimate for prioritizing pages for human review on unseen clients. This is not evidence that a refresh will cause recovery, and the starter label does not establish a clean future outcome.”**

In [4]:
claim_language = {
    "observed": "The label is observed in the starter export.",
    "measured": "Performance is measured with held-out ranking metrics.",
    "directional": "The grouped result is directional for unseen-client prioritization.",
    "decision_support": "The output supports human review; it is not an automatic publishing decision.",
}
for label, sentence in claim_language.items():
    print(f"{label}: {sentence}")

assert random_metrics["Precision@50"] >= 0
assert grouped_metrics["Precision@50"] >= 0
assert "client_grouped" in comparison.index[1]
print("Claim audit passed: wording is limited to observed, measured, directional decision support.")

observed: The label is observed in the starter export.
measured: Performance is measured with held-out ranking metrics.
directional: The grouped result is directional for unseen-client prioritization.
decision_support: The output supports human review; it is not an automatic publishing decision.
Claim audit passed: wording is limited to observed, measured, directional decision support.


## Self-check

- [x] Two paper findings have concrete label and validation questions, framed constructively.
- [x] The same Week-5 model is evaluated before and after changing from row-random to client-grouped validation.
- [x] Base rate, ranking metrics, client overlap, and the grouped-split comparison are visible.
- [x] Leakage checks cover label-derived fields, identifiers, product decisions, and temporal limitations.
- [x] Real grouped-test error cases and claim rewrites use public-safe language.
- [ ] Run every cell top to bottom in Colab, inspect the before/after table, and commit this executed notebook under `work/notebooks/`.